# NB09 — Yoko LAS → DSM registration using QGIS control points

Musanga project root: `/content/drive/MyDrive/Colab Notebooks/Musanga/Final`

- Raw LAS: `Data/LAS/raw/Yoko.las`
- Control pairs: `Data/LAS/raw/Yoko.points`
- Reference DSM: `Data/DSM/Yoko_kvo_georef_DSM.tif`
- Proposed aligned LAS: `Data/Las/Yoko_aligned.las`

**Updated validation (canopy-to-canopy):** both raw and affine-transformed LAS coordinates are rasterized **directly** onto the same DSM-referenced fine grid (0.75 m by default). The reference DSM is block-maximum resampled onto that grid. For both sources, each 3 m comparison cell receives (a) the **maximum fine-cell canopy elevation** and (b) the **95th percentile of fine-cell maxima**. Before/after scores use the *same valid 3 m cells*, with median bias removed for registration dispersion; spatial residual maps accompany histograms.

The 95th percentile uses occupied fine cells, rather than every original LAS return: this mitigates point-density differences but does not eliminate sensor / canopy representation differences. True correspondence still needs spatial overlay review.

**QA first:** `EXPORT_ALIGNED_LAS=False` by default. The affine fit approximates the QGIS point pairs, which may not exactly reproduce a higher-order QGIS warp. **Z is left unchanged.**

In [ ]:
!pip -q install laspy lazrs rasterio pyproj scipy pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import copy, json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import laspy, rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from scipy.ndimage import map_coordinates
from pyproj import CRS

ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
LAS = ROOT/'Data/LAS/raw/Yoko.las'
POINTS = ROOT/'Data/LAS/raw/Yoko.points'
DSM = ROOT/'Data/DSM/Yoko_kvo_georef_DSM.tif'
OUT = ROOT/'Data/Las/Yoko_aligned.las'
QA_DIR = ROOT/'Data/Las'

GRID_RES_M = 3.0
SUBPIXEL_M = 0.75  # 4 x 4 upper-canopy subcells per comparison cell
MAX_FINE_CELLS = 12_000_000
CHUNK_SIZE = 200_000
MIN_VALID_FINE = 4  # required for LAS/DSM 3 m comparison
RNG_SEED = 9

# Review QA and set True in a separate run ONLY if source frame and fit make sense.
EXPORT_ALIGNED_LAS = False
for f in (LAS,POINTS,DSM):
    if not f.is_file(): raise FileNotFoundError(str(f))
QA_DIR.mkdir(parents=True,exist_ok=True)
print('LAS:',LAS,'\nGCP:',POINTS,'\nDSM:',DSM,'\nOutput:',OUT)

In [ ]:
# Read QGIS .points. Its first line is a CRS comment, not a CSV record.
with open(POINTS,encoding='utf-8-sig') as stream:
    first=stream.readline().strip()
gcp=pd.read_csv(POINTS,comment='#')
required={'mapX','mapY','sourceX','sourceY','enable'}
if not required.issubset(gcp): raise ValueError(f'Missing GCP columns: {required-set(gcp)}')
gcp=gcp[gcp.enable.astype(str).str.strip().isin(['1','1.0','True','true'])].copy()
gcp=gcp.replace([np.inf,-np.inf],np.nan).dropna(subset=['mapX','mapY','sourceX','sourceY'])
if len(gcp)<6: raise ValueError('Need at least 6 valid enabled GCPs')
source=gcp[['sourceX','sourceY']].to_numpy(dtype=np.float64)
target=gcp[['mapX','mapY']].to_numpy(dtype=np.float64)
origin=source.mean(axis=0)
A=np.column_stack((source-origin,np.ones(len(source))))
coef=np.linalg.lstsq(A,target,rcond=None)[0]
predict=lambda x,y: ((np.column_stack((np.asarray(x)-origin[0],np.asarray(y)-origin[1],
                                    np.ones_like(np.asarray(x)))) @ coef))
pred=predict(source[:,0],source[:,1])
error=np.linalg.norm(pred-target,axis=1)
rmse=float(np.sqrt(np.mean(error**2)))
print('Enabled GCPs:',len(gcp),'affine horizontal RMSE:',round(rmse,3),'m')
print('Median:',round(np.median(error),3),'m; P95:',round(np.percentile(error,95),3),'m')
print('Linear transform X:',coef[:,0])
print('Linear transform Y:',coef[:,1])
if rmse>3.0:
    warnings.warn('Affine control-point residuals exceed 3 m; inspect QGIS transform type.')
# Ten deterministic random folds for out-of-sample accuracy.
rng=np.random.default_rng(RNG_SEED); order=rng.permutation(len(gcp))
folds=np.array_split(order, min(5,len(gcp)))
cv_error=np.empty(len(gcp))
for test in folds:
    train=np.setdiff1d(np.arange(len(gcp)),test)
    p=np.linalg.lstsq(A[train],target[train],rcond=None)[0]
    cv_error[test]=np.linalg.norm(A[test]@p-target[test],axis=1)
print('5-fold GCP cross-validation RMSE:',np.sqrt(np.mean(cv_error**2)).round(3),'m')
plt.figure(figsize=(6,4))
plt.hist(error,bins=30)
plt.xlabel('Horizontal affine GCP fit error (m)');plt.ylabel('Control points')
plt.title('Yoko QGIS source → map control fit');plt.show()

In [ ]:
with laspy.open(LAS) as src:
    h=src.header
    bounds_min,bounds_max=h.mins.copy(),h.maxs.copy()
    las_crs=h.parse_crs()
    n_points=h.point_count
with rasterio.open(DSM) as ds:
    dsm_crs=ds.crs
    db=ds.bounds
    dres=ds.res
print(f'LAS: {n_points:,} points, bounds',bounds_min,bounds_max,'CRS:',las_crs)
print('DSM:',dsm_crs,'bounds:',db,'resolution:',dres)
if dsm_crs is None or not dsm_crs.is_projected:
    raise ValueError('Reference DSM must have projected metric CRS')
if not np.all(np.isclose([a.unit_conversion_factor for a in CRS.from_user_input(dsm_crs).axis_info[:2]],1)):
    raise ValueError('Expected DSM coordinates in metres.')
if las_crs is not None and not CRS.from_user_input(las_crs).equals(CRS.from_user_input(dsm_crs)):
    raise ValueError('LAS and DSM CRS metadata differ. This affine fit is not CRS reprojection.')
if first.startswith('#CRS:'):
    try:
        qgis_crs=CRS.from_user_input(first[5:].strip())
        if not qgis_crs.equals(CRS.from_user_input(dsm_crs)):
            raise ValueError('QGIS GCP target CRS differs from reference DSM CRS')
    except ValueError:
        raise
# Check that LAS is plausibly in SOURCE (unwarped) coordinates.
las_center=np.array([(bounds_min[0]+bounds_max[0])/2,(bounds_min[1]+bounds_max[1])/2])
source_box=(source.min(0),source.max(0))
map_box=(target.min(0),target.max(0))
print('GCP source bounds:',*source_box)
print('GCP map bounds:',*map_box)
print('LAS centre:',las_center,'mapped centre:',predict([las_center[0]],[las_center[1]])[0])
if not (np.all(las_center>source_box[0]-3000) and np.all(las_center<source_box[1]+3000)):
    raise RuntimeError('LAS centre is far from GCP source envelope. DO NOT transform until coordinate frames are checked.')

In [ ]:
# RAM-bounded, equal-support canopy rasterization.
# Everything is evaluated on a *single* DSM-referenced grid.
# Each 3 m comparison cell contains 4x4 fine (0.75 m) canopy cells.
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling

ratio = int(round(GRID_RES_M / SUBPIXEL_M))
if ratio < 2 or not np.isclose(ratio * SUBPIXEL_M, GRID_RES_M):
    raise ValueError("GRID_RES_M must be an integer multiple of SUBPIXEL_M")
with rasterio.open(DSM) as ds:
    bb = ds.bounds
    x0 = np.floor(bb.left / GRID_RES_M) * GRID_RES_M
    y0 = np.ceil(bb.top / GRID_RES_M) * GRID_RES_M
    coarse_cols = int(np.ceil((bb.right - x0) / GRID_RES_M))
    coarse_rows = int(np.ceil((y0 - bb.bottom) / GRID_RES_M))
    fine_cols = coarse_cols * ratio
    fine_rows = coarse_rows * ratio
    if fine_cols * fine_rows > MAX_FINE_CELLS:
        raise MemoryError(
            f"Need {fine_cols*fine_rows:,} fine cells; raise SUBPIXEL_M "
            "to 1.0 or 1.5 m and ensure it divides GRID_RES_M exactly."
        )
    fine_affine = from_origin(x0, y0, SUBPIXEL_M, SUBPIXEL_M)
    dsm_fine = np.full((fine_rows, fine_cols), np.nan, dtype=np.float32)
    reproject(
        source=rasterio.band(ds,1), destination=dsm_fine,
        src_transform=ds.transform, src_crs=ds.crs, src_nodata=ds.nodata,
        dst_transform=fine_affine, dst_crs=ds.crs, dst_nodata=np.nan,
        resampling=Resampling.max, warp_mem_limit=64, num_threads=2,
        init_dest_nodata=True)
print(f'Fine grid: {fine_rows} x {fine_cols} ({SUBPIXEL_M} m), '
      f'{dsm_fine.nbytes/2**20:.1f} MiB per surface')

# Stream LAS exactly once; for each original point assign the same Z to
# unaligned (source X/Y) and affine-aligned (transformed X/Y) rasterizations.
before_fine = np.full((fine_rows, fine_cols), -np.inf, dtype=np.float32)
after_fine = np.full((fine_rows, fine_cols), -np.inf, dtype=np.float32)
counts = [0,0]
with laspy.open(LAS) as reader:
    for chunk in reader.chunk_iterator(CHUNK_SIZE):
        xs = np.asarray(chunk.x)
        ys = np.asarray(chunk.y)
        zs = np.asarray(chunk.z)
        fitted = predict(xs, ys)
        for k,(xx,yy,arr) in enumerate(
                ((xs,ys,before_fine),(fitted[:,0],fitted[:,1],after_fine))):
            ci = np.floor((xx-x0)/SUBPIXEL_M).astype(np.int64)
            ri = np.floor((y0-yy)/SUBPIXEL_M).astype(np.int64)
            valid = np.isfinite(zs) & (ri>=0) & (ri<fine_rows) & (ci>=0) & (ci<fine_cols)
            counts[k] += int(valid.sum())
            np.maximum.at(arr, (ri[valid],ci[valid]), zs[valid].astype(np.float32))
before_fine[~np.isfinite(before_fine)] = np.nan
after_fine[~np.isfinite(after_fine)] = np.nan
print('Points in DSM grid before/after affine:',counts)

# Aggregate equivalent *fine-cell maxima* into coarse canopy maxima and P95.
# Process row strips so temporary arrays remain small.
def aggregate_canopy(fine, strip_rows=96):
    mx = np.full((coarse_rows,coarse_cols),np.nan,np.float32)
    q95 = np.full_like(mx,np.nan)
    nvalid = np.zeros((coarse_rows,coarse_cols),np.uint8)
    for lo in range(0,coarse_rows,strip_rows):
        hi = min(coarse_rows,lo+strip_rows)
        # Shape: (coarse rows, coarse cols, 16 fine samples)
        block = (fine[lo*ratio:hi*ratio,:]
                 .reshape(hi-lo,ratio,coarse_cols,ratio)
                 .transpose(0,2,1,3).reshape(hi-lo,coarse_cols,ratio**2))
        cnt = np.isfinite(block).sum(axis=2)
        nvalid[lo:hi] = cnt
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            mm = np.nanmax(block,axis=2)
            pp = np.nanpercentile(block,95,axis=2)
        mm[cnt < MIN_VALID_FINE] = np.nan
        pp[cnt < MIN_VALID_FINE] = np.nan
        mx[lo:hi],q95[lo:hi] = mm,pp
    return {'max':mx,'p95':q95,'valid_subcells':nvalid}

surf_dsm=aggregate_canopy(dsm_fine)
surf_before=aggregate_canopy(before_fine)
surf_after=aggregate_canopy(after_fine)
del dsm_fine,before_fine,after_fine
print('Canopy surface aggregation complete.')

# Compare exactly the same spatial 3 m cells in both conditions.
def centered_metrics(residual):
    median = float(np.median(residual))
    centered = residual - median
    return {
        'n':int(len(residual)), 'bias_m':median,
        'centered_MAD_m':float(np.median(np.abs(centered))),
        'centered_RMSE_m':float(np.sqrt(np.mean(centered**2))),
        'p95_abs_centered_m':float(np.percentile(np.abs(centered),95))
    }

results={}
residual_maps={}
for method in ('max','p95'):
    reference=surf_dsm[method]
    raw=surf_before[method]
    aligned=surf_after[method]
    common = np.isfinite(reference)&np.isfinite(raw)&np.isfinite(aligned)
    if int(common.sum())<300:
        raise RuntimeError(f'Too few common canopy comparison cells for {method}: {common.sum()}')
    before_res=reference[common]-raw[common]
    after_res=reference[common]-aligned[common]
    b=centered_metrics(before_res)
    a=centered_metrics(after_res)
    gain=100*(b['centered_MAD_m']-a['centered_MAD_m'])/b['centered_MAD_m']
    results[method]={'before':b,'after':a,'gain_MAD_percent':float(gain)}
    residual_maps[method]={}
    for name,arr in (('before',reference-raw),('after',reference-aligned)):
        resid=np.where(common,arr,np.nan)
        residual_maps[method][name]=resid
    print(f'\n{method.upper()} — SAME {common.sum():,} cells')
    print('Before:',b)
    print('After :',a)
    print(f'Centered MAD improvement: {gain:+.1f}%')
print('Note: negative DSM−LAS median bias does not on its own indicate incorrect XY registration.')

In [ ]:
# Visual QA: spatial residuals plus paired residual histograms.
fig,axes=plt.subplots(2,3,figsize=(17,9))
for row,method in enumerate(('max','p95')):
    layers=residual_maps[method]
    for col,label in enumerate(('before','after')):
        arr=layers[label]
        centered=arr-np.nanmedian(arr)
        im=axes[row,col].imshow(centered,origin='upper',vmin=-12,vmax=12,cmap='RdBu_r')
        axes[row,col].set_title(f'{method.upper()} {label}: centered DSM − LAS Z (m)')
        axes[row,col].set_axis_off()
    fig.colorbar(im,ax=list(axes[row,:2]),fraction=0.025,label='Residual (m)')
    b=layers['before']; a=layers['after']
    m=np.isfinite(b)&np.isfinite(a)
    b=b[m]-np.nanmedian(b[m])
    a=a[m]-np.nanmedian(a[m])
    lo,hi=np.percentile(np.r_[b,a],[1,99])
    axes[row,2].hist(b,bins=75,range=(lo,hi),alpha=.55,label='Before',density=True)
    axes[row,2].hist(a,bins=75,range=(lo,hi),alpha=.55,label='After affine',density=True)
    axes[row,2].axvline(0,color='black',ls='--',lw=1)
    axes[row,2].set_title(f'{method.upper()} — paired cells')
    axes[row,2].set_xlabel('Median-centered DSM − LAS (m)')
    axes[row,2].legend()
plt.subplots_adjust(wspace=.28,hspace=.24)
qa=QA_DIR/'09_Yoko_GCP_canopy_matched_QA.png'
fig.savefig(qa,dpi=160,bbox_inches='tight')
plt.show()

summary={
    'method':'QGIS GCP affine + paired canopy max/p95 validation',
    'GCP_RMSE_m':rmse,'GCP_CV_RMSE_m':float(np.sqrt(np.mean(cv_error**2))),
    'affine_coef_centered':coef.tolist(),'GCP_source_origin_xy':origin.tolist(),
    'grid_m':GRID_RES_M,'fine_grid_m':SUBPIXEL_M,
    'fine_cell_minimum_occupancy':MIN_VALID_FINE,
    'results':results,
    'point_counts_in_reference_extent':counts,
    'export_enabled':EXPORT_ALIGNED_LAS,
    'vertical_shift_applied_m':0
}
report=QA_DIR/'09_Yoko_GCP_canopy_matched_parameters.json'
report.write_text(json.dumps(summary,indent=2))
print('QA:',qa)
print('Parameters:',report)
print('Review both MAX and P95 improvements, map patterns, and GCP residuals before export.')

In [ ]:
# Export only after confirming the source LAS is in the original QGIS SOURCE frame.
# After QA review, change EXPORT_ALIGNED_LAS=True in the configuration cell.
if EXPORT_ALIGNED_LAS:
    if not all(np.isfinite(results[m]['gain_MAD_percent']) and
               results[m]['gain_MAD_percent']>=5 for m in ('max','p95')):
        raise RuntimeError('Export blocked: BOTH paired MAX and P95 comparisons must improve centered MAD >=5%.')
    if rmse>3:
        raise RuntimeError('Export blocked: GCP affine residual RMSE >3 m.')
    if OUT.exists():
        raise FileExistsError(f'{OUT} already exists; never overwrite automatically.')
    temp=OUT.with_name(OUT.stem+'_writing.las')
    if temp.exists():
        raise FileExistsError(f'Partial output exists: {temp}; inspect/remove it first.')
    try:
        with laspy.open(LAS) as reader:
            header=copy.deepcopy(reader.header)
            with laspy.open(temp,mode='w',header=header) as writer:
                for chunk in reader.chunk_iterator(CHUNK_SIZE):
                    xy=predict(np.asarray(chunk.x),np.asarray(chunk.y))
                    chunk.x=xy[:,0];chunk.y=xy[:,1]
                    # Z deliberately unchanged
                    writer.write_points(chunk)
        with laspy.open(temp) as check:
            assert check.header.point_count==n_points
        temp.rename(OUT)
        print('EXPORTED:',OUT)
    except Exception:
        print('Export interrupted. Check temporary LAS:',temp)
        raise
else:
    print('QA ONLY — no new LAS written. Inspect fit, overlap and histograms before enabling export.')

## Review and repeat

The GCP-file affine model changes only XY; elevation and LAS attributes are preserved. The DSM is compared with maximum elevation in 0.75 m cells, and the 3 m **maximum and P95 of those fine-cell maxima**. This is an approximately like-for-like upper-canopy representation, not a direct claim that DSM and LAS have the same acquisition physics.

**A positive paired MAD improvement is necessary, not sufficient:** inspect both residual maps for coherent artifacts and independently examine LAS/DSM overlays in QGIS before accepting the transformation. Keep `EXPORT_ALIGNED_LAS=False` until QA has been reviewed; re-run only the export cell after changing the flag when comfortable. Export never overwrites an existing aligned file.

For other sites, reuse the algorithm with the site's **own** LAS, QGIS point file, georeferenced DSM and output path. Do not reuse Yoko's affine parameters across sites.